# Libraries

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import scripts.yt_utils as yt_utils
import googleapiclient.discovery
from dotenv import load_dotenv
import pandas as pd
import os
import fs

In [19]:
INTERIM_PROJECT_DIR = fs.open_fs("../../data/interim")
YT_VIDEOS_CSV = INTERIM_PROJECT_DIR.getsyspath("YT_videos.csv")
INTERIM_VIDEO_COMMETS_DIR = fs.open_fs("../../data/interim/video_commets")

Read enviromental variables

In [7]:
load_dotenv()

YT_API_KEY = os.getenv('YT-API-KEY')

In [8]:
df = pd.read_csv(YT_VIDEOS_CSV)
df.head()

,publishedAt,channelId,title,description,channelTitle,playlistId,position,videoId,videoOwnerChannelTitle,videoOwnerChannelId
0,2024-10-11T01:21:17Z,UCeYSnd34IPe57sM8hkjWhrw,"Theodore Roosevelt, el gringo más gringo - His...","¡Amigos, acompáñenos el día de hoy a conocer l...",Historia para Tontos Podcast,PLlP9CI1vpjD6siEPRL5TK3S4a6V6KDTIe,0,wCm4FNSnDPs,Historia para Tontos Podcast,UCeYSnd34IPe57sM8hkjWhrw
1,2024-09-26T04:28:43Z,UCeYSnd34IPe57sM8hkjWhrw,Guerra de los 30 años / Paz de Westfalia 🗺 - H...,"¡Amigos, acompáñenos el día de hoy terminaremo...",Historia para Tontos Podcast,PLlP9CI1vpjD6siEPRL5TK3S4a6V6KDTIe,1,5Uac-LKCY4Y,Historia para Tontos Podcast,UCeYSnd34IPe57sM8hkjWhrw
2,2024-09-26T03:36:38Z,UCeYSnd34IPe57sM8hkjWhrw,La Guerra de los 30 años 🗺 - Historia para ton...,"¡Amigos, acompáñenos el día de hoy a explorar ...",Historia para Tontos Podcast,PLlP9CI1vpjD6siEPRL5TK3S4a6V6KDTIe,2,kSRLcVrXYSY,Historia para Tontos Podcast,UCeYSnd34IPe57sM8hkjWhrw
3,2024-09-15T06:45:58Z,UCeYSnd34IPe57sM8hkjWhrw,Maravillas del mundo moderno 🏛 ⛩ - Historia p...,"¡Amigos, acompáñenos el día de hoy a explorar ...",Historia para Tontos Podcast,PLlP9CI1vpjD6siEPRL5TK3S4a6V6KDTIe,3,HxMXLWqe9HQ,Historia para Tontos Podcast,UCeYSnd34IPe57sM8hkjWhrw
4,2024-09-03T02:19:12Z,UCeYSnd34IPe57sM8hkjWhrw,Historia del anime 🗾 - ft César Ruelas - His...,"¡Amigos, acompáñenos el día de hoy a explorar ...",Historia para Tontos Podcast,PLlP9CI1vpjD6siEPRL5TK3S4a6V6KDTIe,4,cOxTBBHDN-0,Historia para Tontos Podcast,UCeYSnd34IPe57sM8hkjWhrw


In [9]:
videoId_list = list(df["videoId"].unique())
print(f"Existen {len(videoId_list)} vídeos únicos")

Existen 25 vídeos únicos


## Pull all videos

In [10]:
videoId_list[0]

'wCm4FNSnDPs'

In [16]:
api_service_name = "youtube"
api_version = "v3"

youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey=YT_API_KEY)

for video in videoId_list:
    YT_VIDEOS_COMMENTS_CSV = INTERIM_VIDEO_COMMETS_DIR.getsyspath(f"video_{video}_commets.csv")
    request = youtube.commentThreads().list(
        part='id,snippet,replies',
        videoId=video
    )
    response = request.execute()
    pd.DataFrame(yt_utils.extract_comments(response)).to_csv(YT_VIDEOS_COMMENTS_CSV, encoding='utf-8', index=False)